# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import torch
import zip

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from hydra.utils import instantiate # noqa: E402
from src.utils.notebook_setup import init_nlp_notebook # noqa: E402

cfg = init_nlp_notebook()
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data & Tokenizer

In [ ]:
from src.core.data import NLPDataModule

# ВАЖНО: Для батчевой генерации паддинг должен быть слева!
cfg.model.tokenizer.padding_side = "left"
tokenizer = instantiate(cfg.model.tokenizer).build()

# Инициализируем модуль данных
datamodule = NLPDataModule(data_cfg=cfg.data, tokenizer=tokenizer)
datamodule.prepare_data()
datamodule.setup(stage="validate")

val_dataloader = datamodule.val_dataloader()
sample_batch = next(iter(val_dataloader))

print("Input IDs shape:", sample_batch["input_ids"].shape)
if "labels" in sample_batch:
    print("Labels contain -100:", (sample_batch["labels"] == -100).any().item())

# Base Model init

In [ ]:
from src.core.models.generation import HFTextGenerator

# Загружаем базовую модель через твой билдер (он сам подхватит квантизацию, если она есть)
model_builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
model = model_builder.build()
model.eval()

print(f"Base Generation Model: {model.config._name_or_path}")

# Инициализируем твой индустриальный генератор со встроенным клинером
generator = HFTextGenerator(
    model=model,
    tokenizer=tokenizer,
    generation_kwargs=cfg.model.generation_kwargs,
    cleaner_cfg=cfg.model.get("cleaner")
)

# Forward Pass (Loss) vs Generation

In [ ]:
from src.core.models.promts import PromptManager

sample_batch = {k: v.to(device) for k, v in sample_batch.items() if isinstance(v, torch.Tensor)}

# 1. Проверка режима "Обучение" (Teacher Forcing для подсчета Perplexity)
with torch.no_grad():
    outputs = model(
        input_ids=sample_batch["input_ids"],
        attention_mask=sample_batch["attention_mask"],
        labels=sample_batch.get("labels")
    )
    perplexity = torch.exp(outputs.loss).item() if outputs.loss is not None else float('nan')
    print(f"Zero-shot Baseline Perplexity (Loss): {perplexity:.2f}")

# 2. Проверка режима "Генерация" (Inference через твой API)
# Берем 2 реальных текста из валидационного датасета
text_col = cfg.data.text_column
raw_texts = datamodule.val_dataset.select(range(2))[text_col]

# Оборачиваем их в промпты через твой менеджер
prompts = [PromptManager.build_simple_prompt(text) for text in raw_texts]

# Генерируем ответы одной строкой (твой генератор сам сделает токенизацию, срежет промпт и почистит мусор!)
responses = generator.generate(texts=prompts, max_new_tokens=50, do_sample=False)

print("\n--- ZERO-SHOT GENERATION EXAMPLE ---")
for p, r in zip(prompts, responses):
    print(f"PROMPT:\n{p}")
    print(f"\nRESPONSE:\n{r}")
    print("-" * 50)

# Baseline Metrics

In [ ]:
import evaluate
from tqdm.auto import tqdm

rouge = evaluate.load('rouge')
all_preds = []
all_refs = [] # Сюда нужно складывать идеальные ответы

# Вытаскиваем нужные колонки из конфига
text_col = cfg.data.text_column
target_col = cfg.data.get("target_column", "summary") # Замени на свою колонку с ответами

# Для примера берем 50 случайных сэмплов из валидации
eval_samples = datamodule.val_dataset.select(range(min(50, len(datamodule.val_dataset))))

# Подготавливаем промпты и эталонные ответы
eval_prompts = [PromptManager.build_simple_prompt(row[text_col]) for row in eval_samples]
all_refs = [row[target_col] for row in eval_samples]

# Генерируем предсказания батчами (например, по 8 штук)
batch_size = 8
for i in tqdm(range(0, len(eval_prompts), batch_size), desc="Zero-shot Generation"):
    batch_prompts = eval_prompts[i : i + batch_size]
    
    # Твой HFTextGenerator работает с батчами из коробки
    batch_preds = generator.generate(batch_prompts, max_new_tokens=50)
    all_preds.extend(batch_preds)

# Считаем ROUGE
results = rouge.compute(predictions=all_preds, references=all_refs)
print("\n--- Baseline ROUGE Scores ---")
for key, value in results.items():
    print(f"{key}: {value:.4f}")